##  Basic Library imports

In [1]:
import os
import pandas as pd 
import numpy as np
import re

##  Read Dataset

In [2]:
DATASET_FOLDER = 'dataset/'
train = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
test = pd.read_csv(os.path.join(DATASET_FOLDER, 'test.csv'))
sample_test = pd.read_csv(os.path.join(DATASET_FOLDER, 'sample_test.csv'))
sample_test_out = pd.read_csv(os.path.join(DATASET_FOLDER, 'sample_test_out.csv'))

train_small = train.head(1000)

In [ ]:
from utils import download_images
download_images(test['image_link'], '../images')

In [3]:
from sentence_transformers import SentenceTransformer

text_embedding_model = SentenceTransformer("google/embeddinggemma-300m")
# text_embedding_model.save("../gemma-300m")

def embed_text(strs):
    return text_embedding_model.encode(strs, show_progress_bar=True)

AcceleratorError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [8]:
import re

def clean_unit(unit_string):
    # 1. Basic Pre-processing: Handle non-string inputs and convert to lowercase
    if not isinstance(unit_string, str):
        return 'unknown'
    
    unit = unit_string.lower().strip()

    # 2. Noise Removal: Get rid of newlines and extra text like "bullet point"
    unit = re.split(r'\n|\bper\b|\bcomes as\b', unit)[0].strip()

    # If a known quantity unit is found, return its standard form.
    extraction_map = {
        'fl oz': r'\b(fl\.?\s*oz\.?|fluid\s*ounce)s?\b',
        'oz': r'\b(oz\.?|ounce)s?\b',
        'lb': r'\b(lb\.?|pound)s?\b',
        'g': r'\b(g|gr|gram|gramm)s?\b',
        'kg': r'\b(kg|kilo)s?\b',
        'ml': r'\b(ml|mililitro|milliliter|millilitre)s?\b',
        'l': r'\b(l|ltr|liter)s?\b',
        'ct': r'\b(ct|count|each|piece|unit|unità)s?\b',
        'gal': r'\b(gal\.?)\b',
        'sq ft': r'\b(sq\s*ft)\b'
    }

    for standard_unit, pattern in extraction_map.items():
        if re.search(pattern, unit):
            return standard_unit

    # Check for empty strings, specific invalid terms, or standalone numbers.
    if unit in ['', 'none', 'product_weight', '---', '-'] or unit.isdigit():
        return 'unknown'

    # If it's not a quantity (from step 3) and not unknown (from step 4),
    return 'other'

def filter_data(data):
    names, image_links, bullet_points, product_descs, units, values, ppu, prices, missings = [], [], [], [], [], [], [], [], []
    for i, row in data.iterrows():
        missing = [0, 0, 0, 0]
        desc = row['catalog_content'].strip().splitlines()

        item_name = desc[0][11:]

        image_link = row['image_link'].split('/')[-1]
        if not os.path.exists("images/"+image_link):
            missing[0] = 1.0

        unit = clean_unit(desc[-1][6:].lower())

        value = float(desc[-2][7:])
        if not value > 0.0:
            value = 1.0
            missing[1] = 1.0
            
        bp = desc[1:-2]
        bullet_point = ""
        product_desc = ""
        for p in bp:
            ps = p.split(": ", 1)
            if len(ps) <= 1:
                continue
            label = ps[0].strip()
            data = ps[1].strip()
            if label == "Product Description":
                product_desc = data
            else:
                bullet_point += data + " | "
        if len(bullet_point) == 0:
            missing[2] = 1.0
        if len(product_desc) == 0:
            missing[3] = 1.0
        missing.append(np.log1p(len(bullet_point)))
        missing.append(np.log1p(len(product_desc)))

        names.append(item_name)
        image_links.append(image_link)
        bullet_points.append(bullet_point)
        product_descs.append(product_desc)
        units.append(unit)
        values.append(value)
        missings.append(missing)
        
        if 'price' in row:
            ppu.append(row['price']/value)
            prices.append(row['price'])
        
    return {'names': names,
            'image_files': image_links,
            'bullet_points': bullet_points,
            'product_descs': product_descs,
            'units': units, 'values': values,
            'ppu': ppu, 'prices': prices,
            'missings': missings}

data_filtered = filter_data(train)
data_filtered_test = filter_data(test)

In [7]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
import pickle

# converts data to a format we can feed to a model. creates embeddings
def process_data(fdata, name_scaler=None, bp_scaler=None, desc_scaler=None):
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    name_embeds = embed_text(fdata['names'])
    bp_embeds = embed_text(fdata['bullet_points'])
    desc_embeds = embed_text(fdata['product_descs'])
    ohe_units = ohe.fit_transform(np.array(fdata['units']).reshape(-1, 1))
    values = np.array(fdata['values'])
    ppu, prices = np.array(fdata['ppu']), np.array(fdata['prices'])
    
    # TODO: check if both of these are needed
    # ppu = np.log1p(ppu)
    prices = np.log1p(prices)
    values = np.log1p(values)
    
    if name_scaler is None:
        name_scaler = StandardScaler()
        name_scaler.fit(name_embeds)
    if bp_scaler is None:
        bp_scaler = StandardScaler()
        bp_scaler.fit(bp_embeds)
    if desc_scaler is None:
        desc_scaler = StandardScaler()
        desc_scaler.fit(desc_embeds)
    
    name_embeds = name_scaler.transform(name_embeds)
    bp_embeds = bp_scaler.transform(bp_embeds)
    desc_embeds = desc_scaler.transform(desc_embeds)
    
    data_X_textembeds = np.column_stack([name_embeds, bp_embeds, desc_embeds])
    data_X_metadata = np.column_stack([ohe_units, values])
    data_y = np.column_stack([ppu, prices])
    return data_X_textembeds, data_X_metadata, data_y, name_scaler, bp_scaler, desc_scaler

In [8]:
processed_data = process_data(data_filtered)
with open(r"../train-processed-gemma.pickle", "wb") as out:
    pickle.dump(processed_data, out)

Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

In [4]:
import pickle

with open(r"../train-processed-gemma.pickle", "rb") as inp:
    processed_data = pickle.load(inp)

print(processed_data[0].shape)

(75000, 2304)


In [ ]:
from torchvision import transforms
from transformers import CLIPProcessor, CLIPModel
import torch
from PIL import Image
# 1. Load the model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

IMAGE_FOLDER = 'images/'
image_files = os.listdir(IMAGE_FOLDER)
print(image_files)
embeddings = []
for i, image_file in enumerate(image_files):
    image_path = os.path.join(IMAGE_FOLDER, image_file)
    image = Image.open(image_path).convert("RGB")
    # 2. Preprocess the image
    inputs = processor(images=image, return_tensors="pt")
    # 3. Move to GPU if available
    if torch.cuda.is_available():
        model = model.to('cuda')
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    else:
        model = model.to('cpu')
        inputs = {k: v.to('cpu') for k, v in inputs.items()}
    # 4. Get the image embeddings
    with torch.no_grad():
        image_features = model.get_image_features(**inputs)

    # 5. Normalize the features (optional but recommended for cosine similarity etc.)
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    print(f"Processed {i+1}/{len(image_files)} images", end='\r')

    embeddings.append(image_features)

#save embeddings as a pickle file
import pickle
with open('embeddings_train.pkl', 'wb') as f:
    pickle.dump(embeddings, f)

import pandas as pd
import os
import numpy as np
import pickle

image_files = os.listdir('images/')
df = pd.read_csv('dataset/train.csv')
with open('embeddings_train.pkl', 'rb') as f:
    embeddings = pickle.load(f)

df['image_filename'] = df['image_link'].apply(lambda x: os.path.basename(x))
embeds = []

for img in df['image_filename']:
    if img in image_files:
        #find index of img in image_files
        index = image_files.index(img)
        print(index)
        embeds.append(embeddings[index])
    else:
        embeds.append(np.zeros((1, 512)))  # or some default value

with open('clip_image_embeddings.pkl', 'wb') as f:
    pickle.dump(embeds, f)

In [66]:
import torch
from torchvision import transforms
from PIL import Image
import tqdm

# --- 1. Setup Model and Transformations ---

def setup_dinov2():
    """
    Loads the DINOv2 model and the necessary image transformations.
    """
    # Check for GPU availability
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load the DINOv2 model (ViT-Base with 14x14 patches)
    # The change is here: 'dinov2_vits14' -> 'dinov2_vitb14'
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').to(device)
    model.eval() # Set the model to evaluation mode

    # DINOv2 uses specific normalization values
    transform = transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    return model, transform, device

# --- 2. Embedding Generation Function ---

def get_image_embeddings(image_paths, modeld):
    model, transform, device = modeld
    all_embeddings = []
    
    with torch.no_grad(): # Disable gradient calculations for inference
        for path in tqdm.tqdm(image_paths, desc="Generating Embeddings"):
            try:
                img = Image.open("../images/"+path).convert('RGB')
                img_tensor = transform(img).unsqueeze(0).to(device)
                embedding = model(img_tensor)
                all_embeddings.append(embedding.cpu())

            except Exception as e:
                print(f"Could not process image {path}: {e}")
                zero = torch.zeros((1, model.embed_dim))
                all_embeddings.append(zero)

    return torch.cat(all_embeddings, dim=0)

for img in data_filtered['image_files']:
    if not os.path.exists("../images"+img):
        print(img)
        break

dinov2 = setup_dinov2()
dino2b_embeds_train = get_image_embeddings(data_filtered['image_files'], dinov2)
with open(r"../train-image-dinov2.pickle", "wb") as out:
    pickle.dump(dino2b_embeds_train, out)

51mo8htwTHL.jpg
Using device: cuda


Using cache found in /home/admini/.cache/torch/hub/facebookresearch_dinov2_main
Generating Embeddings:   0%|          | 99/75000 [00:04<56:20, 22.16it/s]  


KeyboardInterrupt: 

In [67]:
dino2b_embeds_test = get_image_embeddings(data_filtered['image_files'], dinov2)
with open(r"../test-image-dinov2.pickle", "wb") as out:
    pickle.dump(dino2b_embeds_test, out)

Generating Embeddings:  52%|█████▏    | 38949/75000 [34:57<26:50, 22.38it/s]  

Could not process image 51mjZYDYjyL.jpg: [Errno 2] No such file or directory: '../images/51mjZYDYjyL.jpg'


Generating Embeddings:  89%|████████▉ | 66708/75000 [59:55<07:18, 18.91it/s]

Could not process image 91qp2XzOX+L.jpg: [Errno 2] No such file or directory: '../images/91qp2XzOX+L.jpg'


Generating Embeddings: 100%|██████████| 75000/75000 [1:07:19<00:00, 18.57it/s]


In [5]:
from sklearn.preprocessing import StandardScaler

with open('clip_image_embeddings.pkl', "rb") as inp:
    clip_image_embeddings = pickle.load(inp)

img_scaler = StandardScaler()
clip_image_embeddings_scaled = img_scaler.fit_transform(clip_image_embeddings)
print(clip_image_embeddings_scaled.shape)

(75000, 512)


In [13]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torchmetrics.regression import SymmetricMeanAbsolutePercentageError
from tqdm.auto import tqdm

def smape_torch(pred, target, eps=1e-8):
    # pred, target: tensors (same shape). Compute 2*|p - t|/(|p| + |t|)
    num = torch.abs(pred - target)
    den = torch.abs(pred) + torch.abs(target) + eps
    return (2.0 * num / den).mean()

# ----------------- Config / Hyperparams -----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 512            # try 32/64/128 depending on GPU memory
num_epochs = 60            # use early stopping if you want
lr = 0.004742680656440446
weight_decay = 2.4901819097935736e-05
num_workers = 1
pin_memory = True if torch.cuda.is_available() else False
eps = 1e-8

# split to train and test
X_text, X_meta, y_full, _, _, _ = processed_data
X_meta = np.column_stack((X_meta, np.array(data_filtered['missings']))) # todo: do this cleanly
print(X_meta.shape)
y_prices = y_full[:, 1].reshape(-1, 1)
X_text_train, X_text_val, X_meta_train, X_meta_val, X_img_train, X_img_val, y_train, y_val = train_test_split(
    X_text, X_meta, clip_image_embeddings_scaled, y_prices, test_size=0.2
)

# Create DataLoaders
train_dataset = TensorDataset(
    torch.from_numpy(X_text_train).float(),
    torch.from_numpy(X_meta_train).float(),
    torch.from_numpy(X_img_train).float(),
    torch.from_numpy(y_train).float()
)
val_dataset = TensorDataset(
    torch.from_numpy(X_text_val).float(),
    torch.from_numpy(X_meta_val).float(),
    torch.from_numpy(X_img_val).float(),
    torch.from_numpy(y_val).float()
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=pin_memory)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)

# ----------------- Model -----------------
class MultimodalPricePredictor(nn.Module): #default class
    def __init__(self, text_input_dim, meta_input_dim, img_input_dim):
        super().__init__()
        self.text_pathway = nn.Sequential(
            nn.Linear(text_input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3671267503033766),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512)
        )
        self.img_pathway = nn.Sequential(
            nn.Linear(img_input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4009101527664223),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256)
        )
        self.meta_pathway = nn.Sequential(
            nn.Linear(meta_input_dim, 64),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(64),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True)
        )
        self.regression_head = nn.Sequential(
            nn.Linear(512 + 256 + 32, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2515940294686991),
            nn.Linear(256, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, text_input, meta_input, img_input):
        t = self.text_pathway(text_input)
        m = self.meta_pathway(meta_input)
        i = self.img_pathway(img_input)
        fused = torch.cat([t, m, i], dim=1)
        return self.regression_head(fused)
    
model = MultimodalPricePredictor(
    text_input_dim = X_text_train.shape[1],
    meta_input_dim = X_meta_train.shape[1],
    img_input_dim = X_img_train.shape[1]
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=weight_decay, lr=lr)
    

def train_one_model(model, optimizer, seed=42):
    # torch.manual_seed(seed)
    # np.random.seed(seed)


    # model = MultimodalPricePredictor(
    #     text_input_dim = X_text_train.shape[1],
    #     meta_input_dim = X_meta_train.shape[1],
    #     img_input_dim = X_img_train.shape[1]
    # ).to(device)

    # ----------------- Optimizer / Loss / Scheduler -----------------
    criterion = nn.L1Loss()
    # optimizer = torch.optim.AdamW(model.parameters(), weight_decay=weight_decay, lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                        factor=0.5, patience=3)

    # ----------------- Training loop with AMP -----------------
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    best_val_smape = float('inf')
    patience = 7
    stale = 0
    print("Starting training on", device)
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        for text_batch, meta_batch, img_batch, y_batch in train_loader:
            text_batch = text_batch.to(device, non_blocking=True)
            meta_batch = meta_batch.to(device, non_blocking=True)
            img_batch = img_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)   # log1p targets

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                preds_log = model(text_batch, meta_batch, img_batch)   # predicted log1p
                loss = criterion(preds_log, y_batch)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        avg_train_loss = running_loss / len(train_loader)

        # Validation
        model.eval()
        val_losses = 0.0
        val_smape_vals = []
        with torch.no_grad():
            for text_batch, meta_batch, img_batch, y_batch in val_loader:
                text_batch = text_batch.to(device, non_blocking=True)
                meta_batch = meta_batch.to(device, non_blocking=True)
                img_batch = img_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)   # log1p

                preds_log = model(text_batch, meta_batch, img_batch)

                # Loss on log scale (for scheduler)
                val_losses += criterion(preds_log, y_batch).item()

                # Convert to original scale using expm1
                preds_orig = torch.expm1(preds_log.squeeze(1))
                targets_orig = torch.expm1(y_batch.squeeze(1))

                batch_smape = smape_torch(preds_orig, targets_orig, eps=eps)
                
                # i think this happens because some "value" columns are extremely high
                if (torch.isnan(batch_smape)):
                    print(preds_log, targets_orig)
                    break
                
                val_smape_vals.append(batch_smape.item())

        avg_val_loss = val_losses / len(val_loader)
        avg_val_smape = float(np.mean(val_smape_vals))

        # Scheduler step (ReduceLROnPlateau uses validation loss)
        scheduler.step(avg_val_loss)

        print(f"Epoch {epoch:02d} | Train loss: {avg_train_loss:.6f} | Val loss: {avg_val_loss:.6f} | Val sMAPE: {avg_val_smape:.6f}")

        # Simple early stopping on sMAPE
        if avg_val_smape + 1e-12 < best_val_smape:
            best_val_smape = avg_val_smape
            stale = 0
            # save checkpoint
            torch.save(model.state_dict(), f"best_model_seed{seed}.pt")
        else:
            stale += 1
            if stale >= patience:
                print(f"Early stopping (no improvement in {patience} epochs). Best val sMAPE: {best_val_smape:.6f}")
                break

    return best_val_smape

# Done: load best model if you want
# model.load_state_dict(torch.load("best_model_seed{seed}.pt", map_location=device))

(75000, 18)


In [20]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torchmetrics.regression import SymmetricMeanAbsolutePercentageError
from tqdm.auto import tqdm

def smape_torch(pred, target, eps=1e-8):
    # pred, target: tensors (same shape). Compute 2*|p - t|/(|p| + |t|)
    num = torch.abs(pred - target)
    den = torch.abs(pred) + torch.abs(target) + eps
    return (2.0 * num / den).mean()

# ----------------- Config / Hyperparams -----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 512            # try 32/64/128 depending on GPU memory
num_epochs = 60            # use early stopping if you want
lr = 0.004742680656440446
weight_decay = 2.4901819097935736e-05
num_workers = 1
pin_memory = True if torch.cuda.is_available() else False
eps = 1e-8

# split to train and test
X_text, X_meta, y_full, _, _, _ = processed_data
X_meta = np.column_stack((X_meta, np.array(data_filtered['missings']))) # todo: do this cleanly
print(X_meta.shape)
y_prices = y_full[:, 1].reshape(-1, 1)
X_text_train, X_text_val, X_meta_train, X_meta_val, X_img_train, X_img_val, y_train, y_val = train_test_split(
    X_text, X_meta, clip_image_embeddings_scaled, y_prices, test_size=0.2
)

# Create DataLoaders
train_dataset = TensorDataset(
    torch.from_numpy(X_text_train).float(),
    torch.from_numpy(X_meta_train).float(),
    torch.from_numpy(X_img_train).float(),
    torch.from_numpy(y_train).float()
)
val_dataset = TensorDataset(
    torch.from_numpy(X_text_val).float(),
    torch.from_numpy(X_meta_val).float(),
    torch.from_numpy(X_img_val).float(),
    torch.from_numpy(y_val).float()
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=pin_memory)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)

# ----------------- Model -----------------
class MultimodalPricePredictor(nn.Module): #default class
    def __init__(self, text_input_dim, meta_input_dim, img_input_dim):
        super().__init__()
        self.text_pathway = nn.Sequential(
            nn.Linear(text_input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3671267503033766),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512)
        )
        self.img_pathway = nn.Sequential(
            nn.Linear(img_input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4009101527664223),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256)
        )
        self.meta_pathway = nn.Sequential(
            nn.Linear(meta_input_dim, 64),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(64),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True)
        )
        self.regression_head = nn.Sequential(
            nn.Linear(512 + 256 + 32, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2515940294686991)
        )
        self.output_head = nn.Linear(256, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, text_input, meta_input, img_input, return_features=False):
        t = self.text_pathway(text_input)
        m = self.meta_pathway(meta_input)
        i = self.img_pathway(img_input)
        fused = torch.cat([t, m, i], dim=1)
        features = self.regression_head(fused)
        if return_features:
            return features
        return self.output_head(features)

model = MultimodalPricePredictor(
    text_input_dim = X_text_train.shape[1],
    meta_input_dim = X_meta_train.shape[1],
    img_input_dim = X_img_train.shape[1]
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=weight_decay, lr=lr)
    

def train_one_model(model, optimizer, seed=42):
    # torch.manual_seed(seed)
    # np.random.seed(seed)


    # model = MultimodalPricePredictor(
    #     text_input_dim = X_text_train.shape[1],
    #     meta_input_dim = X_meta_train.shape[1],
    #     img_input_dim = X_img_train.shape[1]
    # ).to(device)

    # ----------------- Optimizer / Loss / Scheduler -----------------
    criterion = nn.L1Loss()
    # optimizer = torch.optim.AdamW(model.parameters(), weight_decay=weight_decay, lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                        factor=0.5, patience=3)

    # ----------------- Training loop with AMP -----------------
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    best_val_smape = float('inf')
    patience = 7
    stale = 0
    print("Starting training on", device)
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        for text_batch, meta_batch, img_batch, y_batch in train_loader:
            text_batch = text_batch.to(device, non_blocking=True)
            meta_batch = meta_batch.to(device, non_blocking=True)
            img_batch = img_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)   # log1p targets

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                preds_log = model(text_batch, meta_batch, img_batch)   # predicted log1p
                loss = criterion(preds_log, y_batch)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        avg_train_loss = running_loss / len(train_loader)

        # Validation
        model.eval()
        val_losses = 0.0
        val_smape_vals = []
        with torch.no_grad():
            for text_batch, meta_batch, img_batch, y_batch in val_loader:
                text_batch = text_batch.to(device, non_blocking=True)
                meta_batch = meta_batch.to(device, non_blocking=True)
                img_batch = img_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)   # log1p

                preds_log = model(text_batch, meta_batch, img_batch)

                # Loss on log scale (for scheduler)
                val_losses += criterion(preds_log, y_batch).item()

                # Convert to original scale using expm1
                preds_orig = torch.expm1(preds_log.squeeze(1))
                targets_orig = torch.expm1(y_batch.squeeze(1))

                batch_smape = smape_torch(preds_orig, targets_orig, eps=eps)
                
                # i think this happens because some "value" columns are extremely high
                if (torch.isnan(batch_smape)):
                    print(preds_log, targets_orig)
                    break
                
                val_smape_vals.append(batch_smape.item())

        avg_val_loss = val_losses / len(val_loader)
        avg_val_smape = float(np.mean(val_smape_vals))

        # Scheduler step (ReduceLROnPlateau uses validation loss)
        scheduler.step(avg_val_loss)

        print(f"Epoch {epoch:02d} | Train loss: {avg_train_loss:.6f} | Val loss: {avg_val_loss:.6f} | Val sMAPE: {avg_val_smape:.6f}")

        # Simple early stopping on sMAPE
        if avg_val_smape + 1e-12 < best_val_smape:
            best_val_smape = avg_val_smape
            stale = 0
            # save checkpoint
            torch.save(model.state_dict(), f"best_model_seed{seed}.pt")
        else:
            stale += 1
            if stale >= patience:
                print(f"Early stopping (no improvement in {patience} epochs). Best val sMAPE: {best_val_smape:.6f}")
                break

    return best_val_smape, model

# Done: load best model if you want
# model.load_state_dict(torch.load("best_model_seed{seed}.pt", map_location=device))

(75000, 18)


In [21]:
best_train, final_model = train_one_model(model, optimizer, seed=42)

Starting training on cuda
Epoch 01 | Train loss: 0.919196 | Val loss: 0.568674 | Val sMAPE: 0.566548
Epoch 02 | Train loss: 0.591587 | Val loss: 0.531070 | Val sMAPE: 0.532278
Epoch 03 | Train loss: 0.556566 | Val loss: 0.589807 | Val sMAPE: 0.584477
Epoch 04 | Train loss: 0.541593 | Val loss: 0.513143 | Val sMAPE: 0.514675
Epoch 05 | Train loss: 0.522666 | Val loss: 0.522450 | Val sMAPE: 0.521264
Epoch 06 | Train loss: 0.496694 | Val loss: 0.550491 | Val sMAPE: 0.550258
Epoch 07 | Train loss: 0.482567 | Val loss: 0.500349 | Val sMAPE: 0.503774
Epoch 08 | Train loss: 0.464320 | Val loss: 0.497839 | Val sMAPE: 0.500904
Epoch 09 | Train loss: 0.454358 | Val loss: 0.488406 | Val sMAPE: 0.491868
Epoch 10 | Train loss: 0.438065 | Val loss: 0.488165 | Val sMAPE: 0.490991
Epoch 11 | Train loss: 0.430191 | Val loss: 0.506861 | Val sMAPE: 0.507753
Epoch 12 | Train loss: 0.418743 | Val loss: 0.482435 | Val sMAPE: 0.485680
Epoch 13 | Train loss: 0.404116 | Val loss: 0.478558 | Val sMAPE: 0.481639

In [17]:
num_models = 10
val_scores = []

for seed in range(num_models):
    print(f"\n=== Training model {seed+1}/{num_models} ===")
    smape_val = train_one_model(seed)
    val_scores.append(smape_val)

print("All models trained. Validation sMAPEs:")
print(val_scores)
print(f"Average val sMAPE = {np.mean(val_scores):.6f}")


=== Training model 1/10 ===


TypeError: train_one_model() missing 1 required positional argument: 'optimizer'

In [26]:
import optuna
import os

# ----------------- Configurable Model -----------------
# MODIFICATION: The model now accepts dropout rates and layer sizes as arguments.
class MultimodalPricePredictor(nn.Module):
    def __init__(self, text_input_dim, meta_input_dim, image_input_dim,
                 dropout_text, dropout_image, dropout_head, head_hidden_dim):
        super().__init__()
        self.text_pathway = nn.Sequential(
            nn.Linear(text_input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(dropout_text),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.BatchNorm1d(256)
        )
        self.meta_pathway = nn.Sequential(
            nn.Linear(meta_input_dim, 64), nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.Linear(64, 32), nn.ReLU(inplace=True)
        )
        self.image_pathway = nn.Sequential(
            nn.Linear(image_input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(dropout_image),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.BatchNorm1d(128)
        )
        self.regression_head = nn.Sequential(
            nn.Linear(256 + 32 + 128, head_hidden_dim), nn.ReLU(inplace=True),
            nn.Dropout(dropout_head),
            nn.Linear(head_hidden_dim, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, text_input, meta_input, image_input):
        t, m, i = self.text_pathway(text_input), self.meta_pathway(meta_input), self.image_pathway(image_input)
        fused = torch.cat([t, m, i], dim=1)
        return self.regression_head(fused)
    
def build_model(lr, weight_decay, dropout_text, dropout_image, dropout_head, head_hidden_dim):
    model = MultimodalPricePredictor(
        text_input_dim = X_text_train.shape[1],
        meta_input_dim = X_meta_train.shape[1],
        image_input_dim = X_img_train.shape[1],
        dropout_text = dropout_text,
        dropout_image = dropout_image,
        dropout_head = dropout_head,
        head_hidden_dim = head_hidden_dim
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), weight_decay=weight_decay, lr=lr)
    return model, optimizer 

# ----------------- Optuna Objective Function -----------------
# This function wraps the entire training process for one set of hyperparameters.
def objective(trial: optuna.trial.Trial) -> float:
    # --- Hyperparameter Search Space ---
    # Here we define the ranges for Optuna to search.
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout_text = trial.suggest_float("dropout_text", 0.1, 0.5)
    dropout_image = trial.suggest_float("dropout_image", 0.1, 0.5)
    dropout_head = trial.suggest_float("dropout_head", 0.2, 0.6)
    head_hidden_dim = trial.suggest_categorical("head_hidden_dim", [128, 256, 512])

    model, optimizer = build_model(lr, weight_decay, dropout_text, dropout_image, dropout_head, head_hidden_dim)
    best_val_smape = train_one_model(model, optimizer)

    # Return the final metric for this trial
    return best_val_smape


In [ ]:
study = optuna.create_study(
        direction="minimize",
        # pruner=optuna.pruners.MedianPruner(n_warmup_steps=5) # Pruner to stop unpromising trials
    )
    
# Start the optimization. Optuna will call the 'objective' function n_trials times.
study.optimize(objective, n_trials=50, timeout=3600) # Run for 50 trials or 1 hour

In [30]:
best_params = study.best_params
best_val_smape = study.best_value
best_trial = study.best_trial
print(best_val_smape)
print(best_params)
#save best params to a file
with open("best_params.txt", "w") as f:
    f.write(f"Best sMAPE: {best_val_smape}\n")
    f.write(f"Best params: {best_params}\n")
    f.write(f"Best trial: {best_trial}\n")
    

0.45221699178218844
{'lr': 0.004742680656440446, 'weight_decay': 2.4901819097935736e-05, 'dropout_text': 0.3671267503033766, 'dropout_image': 0.4009101527664223, 'dropout_head': 0.2515940294686991, 'head_hidden_dim': 256}


In [ ]:
processed_test = process_data(data_filtered_test, name_scaler=processed_data[3],
                              bp_scaler=processed_data[4],
                              desc_scaler=processed_data[5])
with open(r"../test-processed-1-minilm-l6.pickle", "wb") as out:
    pickle.dump(processed_test, out)


Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

In [10]:
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# 1. Extract and prepare test data from your dataframe
X_text_test = processed_test[0]
X_meta_test = processed_test[1]

# 3. Create a Test DataLoader
test_dataset = TensorDataset(
    torch.from_numpy(X_text_test).float(),
    torch.from_numpy(X_meta_test).float()
)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 4. Run Inference
model.eval()  # Set the model to evaluation mode
all_predictions = []

with torch.no_grad():  # Disable gradient calculations for efficiency
    for text_batch, meta_batch in tqdm(test_loader, desc="Predicting"):
        text_batch = text_batch.to(device)
        meta_batch = meta_batch.to(device)
        
        # Get model predictions (they are on a log scale)
        preds_log = model(text_batch, meta_batch)
        
        # Convert predictions back to the original price scale
        preds_orig = torch.expm1(preds_log)
        
        # Move predictions to CPU and store them
        all_predictions.append(preds_orig.cpu())

# 5. Combine predictions from all batches
predictions_tensor = torch.cat(all_predictions, dim=0)
predictions_np = predictions_tensor.squeeze().numpy()

print(f"Generated {len(predictions_np)} predictions.")

# The 'predictions_np' array now holds your final price predictions.

Predicting:   0%|          | 0/147 [00:00<?, ?it/s]

Generated 75000 predictions.


In [14]:
out = np.column_stack((np.array(test['sample_id']), predictions_np))
np.savetxt("submission.csv", out, fmt=["%d", "%.8f"], delimiter=",", header="sample_id,price", comments="")